In [1]:
import nilearn as nl
import numpy as np
import matplotlib
import pandas as pd
import os
import nibabel as nib
import matplotlib.pyplot as plt
import openpyxl as xl

from nilearn.glm.first_level import make_first_level_design_matrix
from nilearn.plotting import plot_design_matrix, plot_contrast_matrix
from nilearn.glm.first_level import FirstLevelModel
from nilearn.masking import compute_epi_mask, apply_mask, unmask
from nilearn.image import index_img, resample_to_img, math_img, mean_img, get_data
from nilearn.plotting import plot_stat_map, show
from nilearn import image, masking
from nilearn import plotting
from nilearn.datasets import load_mni152_brain_mask
from pandas import read_excel

from nilearn.glm.second_level import SecondLevelModel
from scipy.stats import norm

from nilearn.glm import threshold_stats_img

In [2]:
textValueFileDF = pd.read_excel("foundationScores/21styear_segment_MFT_MAC.xlsx")

sentenceValues = textValueFileDF[['MAC_a_fairness_virtue',
                               'MAC_a_group_virtue',
                               'MAC_a_deference_virtue',
                               'MAC_a_heroism_virtue',
                               'MAC_a_reciprocity_virtue',
                               'MAC_a_family_virtue',
                               'MAC_a_property_virtue']]



In [3]:
sentenceValues

,MAC_a_fairness_virtue,MAC_a_group_virtue,MAC_a_deference_virtue,MAC_a_heroism_virtue,MAC_a_reciprocity_virtue,MAC_a_family_virtue,MAC_a_property_virtue
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.000000,0.000000,0.025424,0.000000,0.137999,0.000000,0.044444
3,0.000000,0.000000,0.038760,0.030303,0.060870,0.064171,0.051282
4,0.171063,0.184899,0.100000,0.000000,0.093960,0.065217,0.103700
...,...,...,...,...,...,...,...
792,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.021739
793,0.000000,0.030864,0.003460,0.028455,0.008304,0.036036,0.012625
794,0.000000,0.100000,0.042904,0.000000,0.033824,0.000000,0.034483
795,0.083651,0.096163,0.009259,0.021609,0.013619,0.229843,0.006039


In [4]:

events = pd.DataFrame( {"trial_type":range(0,len(sentenceValues))})

# dtermine the max column for each row
max_column = sentenceValues.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (sentenceValues.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list

In [5]:
events

,trial_type
0,baseline
1,baseline
2,MAC_a_reciprocity_virtue
3,MAC_a_family_virtue
4,MAC_a_group_virtue
...,...
792,MAC_a_property_virtue
793,MAC_a_family_virtue
794,MAC_a_group_virtue
795,MAC_a_family_virtue


In [6]:
baseline_count = (events['trial_type'] == 'baseline').sum()
print('number of baseline: ',baseline_count)

fai_count = (events['trial_type'] == 'MAC_a_fairness_virtue').sum()
print('number of fairness virtue: ',fai_count)

fam = (events['trial_type'] == 'MAC_a_family_virtue').sum()
print('number of family virtue: ',fam)

her_count = (events['trial_type'] == 'MAC_a_heroism_virtue').sum()
print('number of heroism virtue: ',her_count)

rec_count = (events['trial_type'] == 'MAC_a_reciprocity_virtue').sum()
print('number of reciprocity virtue: ',rec_count)

gr_count = (events['trial_type'] == 'MAC_a_group_virtue').sum()
print('number of group virtue: ',gr_count)

prop_count = (events['trial_type'] == 'MAC_a_property_virtue').sum()
print('number of property virtue: ',prop_count)

def_count = (events['trial_type'] == 'MAC_a_deference_virtue').sum()
print('number of deference virtue: ',def_count)

number of baseline:  207
number of fairness virtue:  80
number of family virtue:  137
number of heroism virtue:  53
number of reciprocity virtue:  90
number of group virtue:  147
number of property virtue:  27
number of deference virtue:  56


In [7]:
sentenceValues_vice = textValueFileDF[['MAC_a_fairness_vice',
                               'MAC_a_group_vice',
                               'MAC_a_deference_vice',
                               'MAC_a_heroism_vice',
                               'MAC_a_reciprocity_vice',
                               'MAC_a_family_vice',
                               'MAC_a_property_vice']]


events = pd.DataFrame( {"trial_type":range(0,len(sentenceValues_vice))})

# dtermine the max column for each row
max_column = sentenceValues_vice.idxmax(axis=1)

# Determine rows where all 7 columns are 0.00 (to two decimal places)
all_zero_mask = (sentenceValues_vice.round(2) == 0).all(axis=1)
# Replace column name with "baseline" for those rows
max_column[all_zero_mask] = "baseline"

max_columns_list = max_column.tolist()

events['trial_type'] = max_columns_list

In [8]:
baseline_count = (events['trial_type'] == 'baseline').sum()
print('number of baseline: ',baseline_count)

fai_count = (events['trial_type'] == 'MAC_a_fairness_vice').sum()
print('number of fairness vice: ',fai_count)

fam = (events['trial_type'] == 'MAC_a_family_vice').sum()
print('number of family vice: ',fam)

her_count = (events['trial_type'] == 'MAC_a_heroism_vice').sum()
print('number of heroism vice: ',her_count)

rec_count = (events['trial_type'] == 'MAC_a_reciprocity_vice').sum()
print('number of reciprocity vice: ',rec_count)

gr_count = (events['trial_type'] == 'MAC_a_group_vice').sum()
print('number of group vice: ',gr_count)

prop_count = (events['trial_type'] == 'MAC_a_property_vice').sum()
print('number of property vice: ',prop_count)

def_count = (events['trial_type'] == 'MAC_a_deference_vice').sum()
print('number of deference vice: ',def_count)

number of baseline:  215
number of fairness vice:  186
number of family vice:  56
number of heroism vice:  85
number of reciprocity vice:  81
number of group vice:  71
number of property vice:  43
number of deference vice:  60
